In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q2-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
import os
import pandas as pd
import numpy as np
from PIL import Image
from sklearn.model_selection import train_test_split

# Load Labels
labels_df = pd.read_csv(os.path.join(path, "labels.csv"))
img_dir = os.path.join(path, "images")

images = []
ages = []

# Load Images and Resize
print("Loading images...")
for index, row in labels_df.iterrows():
    img_name = row.iloc[0]
    age = row.iloc[1]
    img_path = os.path.join(img_dir, img_name)

    if os.path.exists(img_path):
        img = Image.open(img_path).convert('RGB')
        img_array = np.array(img) / 255.0 # Normalize pixel values to [0, 1]
        images.append(img_array)
        ages.append(age)

X = np.array(images)
y = np.array(ages)

# Transpose image dimensions to match PyTorch format (N, C, H, W)
X = np.transpose(X, (0, 3, 1, 2))

# Split Data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

In [ ]:
# 1. Convert Numpy arrays to PyTorch Tensors
import torch

X_train_t = torch.tensor(X_train, dtype=torch.float32)
X_test_t  = torch.tensor(X_test, dtype=torch.float32)

y_train_t = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)
y_test_t  = torch.tensor(y_test, dtype=torch.float32).unsqueeze(1)




In [ ]:
# 2. Create TensorDataset objects
# Task 2: Create dataset
from torch.utils.data import TensorDataset

train_dataset = TensorDataset(X_train_t, y_train_t)
test_dataset  = TensorDataset(X_test_t, y_test_t)

In [ ]:
# 3. Create DataLoaders
from torch.utils.data import DataLoader

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader  = DataLoader(test_dataset, batch_size=32, shuffle=False)



In [ ]:
# 4. Print shape of one batch
xb, yb = next(iter(train_loader))
print("X batch:", xb.shape)
print("y batch:", yb.shape)




In [ ]:
# 5. Display sample images
import matplotlib.pyplot as plt

xb, yb = next(iter(train_loader))

plt.figure(figsize=(10, 4))
for i in range(6):
    img = xb[i].permute(1, 2, 0).numpy()
    plt.subplot(2, 3, i+1)
    plt.imshow(img)
    plt.title(f"Age: {yb[i].item():.0f}")
    plt.axis("off")
plt.tight_layout()
plt.show()




In [ ]:

# Task 1: Write your model class here:
import torch
import torch.nn as nn

class AgeRegressor(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Flatten(),
            nn.LazyLinear(512),
            nn.ReLU(),
            nn.Linear(512, 128),
            nn.ReLU(),
            nn.Linear(128, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        return self.model(x)


In [ ]:
# Task 2: Write your training loop here:
def train_one_epoch(model, loader, loss_fn, optimizer, device):
    model.train()
    loss = 0

    for X, y in loader:
        X, y = X.to(device), y.to(device)
        optimizer.zero_grad()
        output = model(X)
        l = loss_fn(output, y)
        l.backward()
        optimizer.step()
        loss += l.item()

    return loss / len(loader)

In [ ]:
# Task 3: Write your validation loop here:
def validate(model, loader, loss_fn, device):
    model.eval()
    loss = 0

    for X, y in loader:
        X, y = X.to(device), y.to(device)
        output = model(X)
        loss += loss_fn(output, y).item()

    return loss / len(loader)



In [ ]:
# Task 4: Define device, model, loss, optimizer:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = AgeRegressor().to(device)
loss_fn = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)


In [ ]:
# Task 5: Start training for 20 epochs:

# Create loss lists
train_losses = []
val_losses = []

# Train model
for epoch in range(20):
    train_loss = train_one_epoch(model, train_loader, loss_fn, optimizer, device)
    val_loss = validate(model, test_loader, loss_fn, device)

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    print(epoch + 1)



In [ ]:
# Task 1: Write your code here:
import matplotlib.pyplot as plt

plt.plot(train_losses)
plt.plot(val_losses)
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend(["Train", "Val"])
plt.show()


In [ ]:
# Task 2 (Bonus): Write your code here:
import matplotlib.pyplot as plt
import torch

model.eval()
X, y = next(iter(test_loader))
X = X.to(device)

with torch.no_grad():
    pred = model(X).cpu()

for i in range(6):
    plt.subplot(2, 3, i+1)
    plt.imshow(X[i].cpu().permute(1, 2, 0))
    plt.title(f"P:{pred[i].item():.1f} T:{y[i].item():.0f}")
    plt.axis("off")

plt.show()
